# rhino-poc — Colab kurulum ve calistirma

**Baslamadan:** Runtime > Change runtime type > **GPU (T4)** sec.

**Bir kere yapilacak:** `rhino-poc` klasorunu Drive'ina `MyDrive/rhino-poc` olarak yukle.

Hucreleri sirayla kos. COLMAP kurulumu ilk oturumda uzun surebilir (kaynaktan derleme yolu ~30-45 dk) ama sonucu Drive'a cache'lenir; sonraki oturumlarda ~1 dk.

**Colab kisitlari (bilerek kabul ettik):** oturum kapaninca disk silinir (bu yuzden her sey Drive'a yazilir), uzun MVS kosularinda zaman asimi riski var, Docker yok. Rakamlar ciddiye binince ayni repo kiralik GPU'ya tasinabilir — kod degismez.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
REPO = '/content/drive/MyDrive/rhino-poc'
CACHE = '/content/drive/MyDrive/rhino-poc-cache'
!mkdir -p {CACHE}

## COLMAP (CUDA'li) kurulumu
Once hizli yolu dene (conda-forge GPU derlemesi). `colmap -h` calisiyorsa ve asagidaki dogrulama hucresi CUDA gosteriyorsa bir sonraki bolume gec; calismiyorsa kaynaktan derleme hucresini kos.

In [ ]:
%%bash
# Hizli yol: micromamba + conda-forge GPU derlemesi (~2-4 dk)
cd /opt
curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba
/opt/bin/micromamba create -y -q -p /opt/colmapenv -c conda-forge "colmap=*=gpu*" || \
  echo 'GPU derlemesi bulunamadi — asagidaki kaynaktan derleme hucresini kos.'

In [ ]:
# COLMAP'i PATH'e SYMLINK ile ver (PATH'in basina conda klasoru EKLEME:
# icindeki python, sistemin python'unu golgeleyip cv2 hatasina yol acar).
import os
for c in ('/opt/colmapenv/bin/colmap', '/usr/local/colmap/bin/colmap'):
    if os.path.exists(c):
        os.system(f'ln -sf {c} /usr/local/bin/colmap')
        break
# yanlislikla eklenmisse temizle:
os.environ['PATH'] = os.environ['PATH'].replace('/opt/colmapenv/bin:', '')
!colmap -h 2>/dev/null | head -3 || echo 'colmap bulunamadi'

In [ ]:
%%bash
# YEDEK YOL — SADECE yukaridaki dogrulama 'colmap bulunamadi' derse kos!
# (Ustteki hucre 'with CUDA' gosterdiyse bu hucreyi ATLA.)
# Kaynaktan derleme: ilk seferde ~30-45 dk, Drive'a cache'lenir.
# CUDA_ARCH: T4=75, L4=89, A100=80 (nvidia-smi ciktisina gore ayarla)
apt-get update -q > /dev/null   # apt 404 hatalarini onler
CACHE=/content/drive/MyDrive/rhino-poc-cache
if [ -f $CACHE/colmap-cuda.tar.gz ]; then
  echo 'Cache bulundu, aciliyor...'
  tar -xzf $CACHE/colmap-cuda.tar.gz -C /usr/local
else
  apt-get install -y -q libboost-all-dev libeigen3-dev libflann-dev libfreeimage-dev \
    libmetis-dev libgoogle-glog-dev libsqlite3-dev libglew-dev libceres-dev \
    qtbase5-dev libqt5opengl5-dev libcgal-dev > /dev/null
  git clone --depth 1 https://github.com/colmap/colmap.git /tmp/colmap
  cd /tmp/colmap && mkdir -p build && cd build
  cmake .. -DCMAKE_BUILD_TYPE=Release -DCUDA_ENABLED=ON \
    -DCMAKE_CUDA_ARCHITECTURES=75 -DGUI_ENABLED=OFF \
    -DCMAKE_INSTALL_PREFIX=/usr/local/colmap > /dev/null
  make -j$(nproc) > /dev/null && make install > /dev/null
  tar -czf $CACHE/colmap-cuda.tar.gz -C /usr/local colmap
fi
/usr/local/colmap/bin/colmap -h | head -3

## Python bagimliliklari + repo kurulumu

In [ ]:
!pip install -q opencv-contrib-python open3d trimesh pycolmap typer pandas pillow
# NOT: '-e' (editable) KULLANMA — Jupyter'de ayni oturumda import edilemez (restart ister).
# Normal kurulum kodu kopyalar; repo'da kod degistirirsen bu hucreyi tekrar kos.
!pip install -q {REPO}
import poc; print('poc', poc.__version__, 'hazir')

## ArUco marker uret (bir kere)
PNG'yi indir, **'Gercek boyut / %100'** ile MAT kagida bas, alttaki cizginin 100 mm oldugunu cetvelle dogrula, **marker kenarini olc** ve asagidaki `MARKER_MM`'e yaz.

In [ ]:
!python {REPO}/scripts/make_aruco.py --mm 50 --out /content/aruco_50mm.png
from google.colab import files; files.download('/content/aruco_50mm.png')

## Video isle
Videoyu Drive'a at (orn. `MyDrive/rhino-poc-data/vaka_001.mp4`) ve calistir. Cikti Drive'a yazilir; oturum duserse kaybolmaz.

In [ ]:
VIDEO = '/content/drive/MyDrive/rhino-poc-data/vaka_001.mp4'
OUT   = '/content/drive/MyDrive/rhino-poc-data/vaka_001'
MARKER_MM = 50.0   # BASILI marker'in cetvelle olculmus kenari!

!poc process {VIDEO} --out {OUT} --marker-mm {MARKER_MM}

## Sonuclar
`OUT` klasorunde: `frames/` secilen kareler, `colmap/` SfM+MVS ara ciktilar, `mesh_raw.ply`, `scale.json`, `model.glb` (mm olcekli — [gltf.report](https://gltf.report) veya Blender'da ac, bbox ~200-250 mm kafa boyunda olmali).

Ilk kontrol: `scale.json` icindeki `side_spread_pct` < 5 olmali; `model.glb`'de iki gozbebegi arasi tikla-olc ~60-70 mm cikmali (makullluk).